## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import sys, os

DATA_PATH = '../data/final_eliza.csv'

LABEL_NAMES = {
    0: 'Sadness', 1: 'Joy',  2: 'Love',
    3: 'Anger',   4: 'Fear', 5: 'Surprise'
}

COLORS = {
    'Sadness' : '#5B8DB8',
    'Joy'     : '#F4C542',
    'Love'    : '#E8727A',
    'Anger'   : '#D94F3D',
    'Fear'    : '#8B6BB1',
    'Surprise': '#4BAE8A'
}

sns.set_theme(style='whitegrid')
print('Setup complete!')

### Load Data


In [ ]:
df = pd.read_csv(DATA_PATH)
df.columns  = df.columns.str.strip()
df['text']  = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df['emotion'] = df['label'].map(LABEL_NAMES)

print(f'Total rows : {len(df)}')
print(f'Columns    : {list(df.columns)}')
df.head(10)

## Class Distribution

In [ ]:
counts = df['emotion'].value_counts().reindex(LABEL_NAMES.values())

print('Class Distribution:')
print('-' * 45)
for emotion, count in counts.items():
    bar = '█' * (count // 5)
    pct = count / len(df) * 100
    print(f'{emotion:<10} {count:>4} rows ({pct:.1f}%)  {bar}')
print('-' * 45)
print(f'Total      {len(df):>4} rows')

ratio = counts.max() / counts.min()
print(f'\nMax/Min ratio: {ratio:.2f}x')
if ratio > 2.0:
    print('⚠️  Imbalanced — collect more data for smaller classes')
else:
    print('✅ Classes are balanced')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Class Distribution — Burmese Emotion Dataset', fontsize=14, fontweight='bold')

bar_colors = [COLORS[e] for e in counts.index]

# Bar chart
axes[0].bar(counts.index, counts.values, color=bar_colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Row Count per Class')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Count')
for i, (emotion, count) in enumerate(counts.items()):
    axes[0].text(i, count + 2, str(count), ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylim(0, counts.max() * 1.15)

# Pie chart
axes[1].pie(
    counts.values, labels=counts.index,
    colors=bar_colors, autopct='%1.1f%%',
    startangle=140, pctdistance=0.85
)
axes[1].set_title('Class Proportion')

plt.tight_layout()
plt.savefig('../data/eda_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

### Text Length

In [ ]:
sys.path.insert(0, os.path.abspath('.'))
try:
    from burmese_tokenizer import tokenize
    df['token_len'] = df['text'].apply(lambda t: len(tokenize(str(t))))
    print('[OK] Using burmese_tokenizer')
except ImportError:
    df['token_len'] = df['text'].apply(lambda t: len(str(t)))
    print('[Fallback] Using char count')

df['char_len'] = df['text'].apply(len)

stats = df.groupby('emotion')['token_len'].agg(['min','max','mean','median']).round(1)
stats.columns = ['Min', 'Max', 'Mean', 'Median']
print('\nToken Length per Class:')
print(stats)

too_short = df[df['token_len'] <= 1]
too_long  = df[df['token_len'] >= 80]
print(f'\n⚠️  Very short (≤1 token) : {len(too_short)} rows')
print(f'⚠️  Very long  (≥80 tokens): {len(too_long)} rows')
if len(too_short) > 0:
    print(too_short[['text','emotion','token_len']].head())

### Text length (chart)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Text Length Analysis', fontsize=14, fontweight='bold')

emotion_order  = list(LABEL_NAMES.values())
data_by_class  = [df[df['emotion'] == e]['token_len'].values for e in emotion_order]

bp = axes[0].boxplot(
    data_by_class, labels=emotion_order,
    patch_artist=True,
    medianprops=dict(color='black', linewidth=2)
)
for patch, emotion in zip(bp['boxes'], emotion_order):
    patch.set_facecolor(COLORS[emotion])
    patch.set_alpha(0.7)
axes[0].set_title('Token Length per Class')
axes[0].set_xlabel('Emotion')
axes[0].set_ylabel('Token Count')

axes[1].hist(df['token_len'], bins=30, color='#5B8DB8', edgecolor='white')
axes[1].axvline(df['token_len'].mean(),   color='red',    linestyle='--', label=f'Mean: {df["token_len"].mean():.1f}')
axes[1].axvline(df['token_len'].median(), color='orange', linestyle='--', label=f'Median: {df["token_len"].median():.1f}')
axes[1].set_title('Overall Token Length')
axes[1].set_xlabel('Token Count')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_text_length.png', dpi=150, bbox_inches='tight')
plt.show()

## Duplicate detect

In [ ]:
total      = len(df)
exact_dups = df.duplicated(subset=['text'], keep=False)
n_exact    = exact_dups.sum()

print(f'Total rows       : {total}')
print(f'Exact duplicates : {n_exact} ({n_exact/total*100:.1f}%)')

if n_exact > 0:
    print('\nSample duplicates:')
    print(df[exact_dups].sort_values('text')[['text','emotion']].head(10).to_string(index=False))

# Cross-class duplicates — most dangerous (same text, different label)
cross = df[df.duplicated(subset=['text'], keep=False)]
cross = cross.groupby('text')['emotion'].nunique()
cross = cross[cross > 1]
print(f'\nCross-class duplicates (label errors): {len(cross)}')
if len(cross) > 0:
    for text in cross.index[:5]:
        labels = df[df['text'] == text]['emotion'].tolist()
        print(f'  "{text}" → {labels}')

### Clean&Save

In [ ]:
print('Duplicates per class:')
print('-' * 40)
for emotion in LABEL_NAMES.values():
    class_df = df[df['emotion'] == emotion]
    n_dups   = class_df.duplicated(subset=['text']).sum()
    print(f'{emotion:<10} : {n_dups:>3} duplicates / {len(class_df)} rows')

df_clean = df.drop_duplicates(subset=['text']).reset_index(drop=True)
print(f'\nBefore : {len(df)} rows')
print(f'After  : {len(df_clean)} rows')
print(f'Removed: {len(df) - len(df_clean)} duplicates')

clean_path = '../data/emotions_my_clean.csv'
df_clean[['text','label']].to_csv(clean_path, index=False, encoding='utf-8-sig')
print(f'\n✅ Saved → {clean_path}')
print(f'\nRetrain with:')
print(f'  python hybrid-eliza.py --mode train --data ../data/emotions_my_clean.csv')